<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/DistilBert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we will explore the mechanisms underlying BERT such as multi-head attention and visualize their workings. We will then perform transfer learning by freezing BERT layers and importing some basic BERT models.

In [ ]:
!git clone https://github.com/brianellis1997/Sarcasm_Detection.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install bertviz

In [ ]:
from transformers import BertModel, BertTokenizer
from bertviz import head_view
import torch

In [ ]:
import pandas as pd
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

# Visualizing Multi-Head Attention

In [ ]:
# Let's load a vanilla BERT-base model.
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained('bert-base-cased')

In [ ]:
sample_text = train_bal['comment'][40]
sample_text

In [ ]:
tokens = tokenizer.encode(sample_text)
inputs = torch.tensor(tokens).unsqueeze(0) # unsqueeze changes the shape from (20,) -> (1, 20)
inputs

In [ ]:
attention = model(inputs, output_attentions=True)[2]  # Grab the attention scores from BERT

In [ ]:
# average attention in the last encoder
final_attention = attention[-1].mean(1)[0]
attention_df = pd.DataFrame(final_attention.detach()).applymap(float).round(3)

In [ ]:
attention_df.columns = tokenizer.convert_ids_to_tokens(tokens)
attention_df.index = tokenizer.convert_ids_to_tokens(tokens)

attention_df  # sums across rows add up to 1. sums across columns do not

In [ ]:
# Layer index 2 seems to be attending to the previous token
# Layer index 6 seems to be for pronouns

tokens_as_list = tokenizer.convert_ids_to_tokens(inputs[0])
head_view(attention, tokens_as_list)

In [ ]:
# Head 3-1 attends to previous token
head_view(attention, tokenizer.convert_ids_to_tokens(inputs[0]), layer=2, heads=[0])

In [ ]:
# Head 8-10 relating direct objects to their verbs eg told -> me
head_view(attention, tokenizer.convert_ids_to_tokens(inputs[0]), layer=7, heads=[9])

In [ ]:
# attention in the 8th encoder's 10th head to see direct object attention
eight_ten = attention[7][0][9]

In [ ]:
# Get the attention matrix
attention_df = pd.DataFrame(eight_ten.detach()).applymap(float).round(3)

attention_df.columns = tokenizer.convert_ids_to_tokens(tokens)
attention_df.index = tokenizer.convert_ids_to_tokens(tokens)

attention_df  # sums across rows add up to 1. sums across columns do not

## BERT-base

The BERT-base model we have been using has 12 stacked encoders containing 110M learnable parameters. There are other BERT sizes we might experiment with such as BERT-small which contains 4 encoders or BERT-large which contains 24. Our usage of these different sizes will depend on the processing speed of BERT-base and how fast it can work with our dataset.

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model = BertModel.from_pretrained('bert-base-cased')  # we are using cased BERT because of our previous analysis showing case utility in detecting sarcasm

In [ ]:
model #inspect the model

In [ ]:
# Get all of the model's parameters as a list of tuples.
named_params = list(model.named_parameters())

print('The BERT model has {:} different named parameters.\n'.format(len(named_params)))

print('==== Embedding Layer ====\n')
for p in named_params[0:5]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== First Encoder ====\n')
for p in named_params[5:21]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== Output Layer ====\n')
for p in named_params[-2:]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

In [ ]:
# load the bert-base uncased tokenizer.
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

In [ ]:
print(f'Original comment: {train_bal.comment[40]}\nTokenization using BERT: {tokenizer.encode(train_bal.comment[40])}')

In [ ]:
# run tokens through the model

#1 Turn tokens_with_unknown_words into a tensor (will be size (8,))
#2 Unsqueeze a first dimension to simulate batches. Resulting shape is (1, 8)
response = model(torch.tensor(tokenizer.encode(train_bal.comment[40])).unsqueeze(0))

In [ ]:
response

In [ ]:
# Embedding for each token, the first one being the [CLS] token
response.last_hidden_state

In [ ]:
# This layer is trained on top of the Embedding of the CLS token

response.pooler_output.shape

In [ ]:
model.pooler

In [ ]:
# grab the final encoder's representation of the CLS token
CLS_embedding = response.last_hidden_state[:, 0, :].unsqueeze(0)

CLS_embedding.shape

We can see that BERT-base embeddings have 768 dimensions

In [ ]:
example_phrase = tokenizer.encode(train_bal.comment[40], return_tensors='pt')
example_phrase

In [ ]:
# context-less embedding of each token in our sentence
model.embeddings.word_embeddings(example_phrase)

In [ ]:
model.embeddings.word_embeddings(example_phrase).shape

Each word in each comment will be tokenized and that token will be represented by a row containing 768 columns.

### Tokenization

In [ ]:
# Look at length of our BERT-base vocabulary
len(tokenizer.vocab)

In [ ]:
# Find the index of the longest comment
longest_index = train_bal['comment'].str.len().idxmax()

# Retrieve the longest comment using the index
longest_com = train_bal.loc[longest_index, 'comment']

longest_com

In [ ]:
longest_token = tokenizer.encode(longest_com)
longest_index

We see that BERT's tokenizer has max limits for tokens of 512.

In [ ]:
# Filter the DataFrame to include only rows where the length of the comment is greater than 500
filtered_train_bal = train_bal[train_bal['comment'].str.len() > 500]

# Calculate the number of rows in the filtered DataFrame
num_rows_with_long_comments = len(filtered_train_bal)

num_rows_with_long_comments

These 235 rows might present problems for our BERT model. Let's see the distribution of label.

In [ ]:
filtered_train_bal['label'].value_counts(normalize=True)

A significant amount of them, 75%, were non-sarcastic

## Demonstrate some Downstream Tasks

### Masked Language Modeling

In [ ]:
from transformers import BertForMaskedLM, pipeline

In [ ]:
bert_lm = BertForMaskedLM.from_pretrained('bert-base-cased')
bert_lm

In [ ]:
nlp = pipeline('fill-mask', model='bert-base-cased')

In [ ]:
nlp.tokenizer

In [ ]:
preds = nlp(f"Yes, lets have people {nlp.tokenizer.mask_token} near a person who could possibly have ebola")  # real word is 'nap'

print('Yes, lets have people *** near a person who could possibly have ebola')

for p in preds:
  print(f"Token: {p['token_str']}. Score: {100*p['score']:,.2f}%")

### Next Sentence Prediction

Our dataset is unique in that it contains not only the sarcastic or non-sarcastic comment, but also the parent comment which preceded it. BERT contains next sentence prediction (nsp) capability which might help us exploit this relationship and improve modeling capability.

In [ ]:
from transformers import BertForNextSentencePrediction, BertTokenizer
import torch

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

bert_nsp = BertForNextSentencePrediction.from_pretrained('bert-base-cased')

In [ ]:
bert_nsp

In [ ]:
import numpy as np
parent_com = train_bal['parent_comment'][40]
com = train_bal['comment'][40]

print(parent_com)
print(com)

In [ ]:
inputs = tokenizer(parent_com, com, return_tensors='pt')
inputs

In [ ]:
# 0 == next sentence and 1 == not next sentence
outputs = bert_nsp(**inputs)

outputs

In [ ]:
outputs = bert_nsp(**inputs, labels=torch.LongTensor([0]))
outputs

The loss is very low which means our bert base model is very confident that these two sentences are connected which is good. This connection between parent comment and child comment might help us classify things better with this BERT capability.<br>We can analyze the places where the nsp confidence is low and see if there's any correlation between confidence and label.

# Sequence Classification

In [ ]:
!pip install datasets

In [ ]:
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, DistilBertTokenizerFast, \
     DataCollatorWithPadding, pipeline
from datasets import load_metric, Dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [ ]:
# conjoin comment and parent comment
train_bal['joined_comment'] = 'parent_comment: ' + train_bal['parent_comment'] + ' child_comment: ' + train_bal['comment']
# here we are concatenating the comment to the parent comment and providing context for our LLM model by prompting the parent comment and child comment locations.

train_bal.head()

In [ ]:
train_bal['joined_comment'][0]

In [ ]:
# Filter the DataFrame to include only rows where the length of the comment is greater than 500
filtered_train_bal = train_bal[train_bal['joined_comment'].str.len() > 500]

# Calculate the number of rows in the filtered DataFrame
num_rows_with_long_comments = len(filtered_train_bal)

num_rows_with_long_comments

In [ ]:
# This code segment parses our dataset into a more manageable format

utterances = train_bal['joined_comment']
sequence_labels = train_bal['label']

In [ ]:
len(utterances), len(sequence_labels)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-cased')

In [ ]:
unique_sequence_labels = list(set(sequence_labels))
unique_sequence_labels

In [ ]:
sequence_labels = [unique_sequence_labels.index(l) for l in sequence_labels]

print(f'There are {len(unique_sequence_labels)} unique sequence labels')

In [ ]:
sarc_dataset = Dataset.from_dict(
    dict(
        utterance=utterances,
        label=sequence_labels
    )
)
sarc_dataset = sarc_dataset.train_test_split(test_size=0.2)

In [ ]:
sarc_dataset['train'][0]

In [ ]:
# simple function to batch tokenize utterances with truncation
def preprocess_function(examples):
    return tokenizer(examples["utterance"], truncation=True)  # Truncation will solve the problem of longer than 512 length tokens

In [ ]:
seq_clf_tokenized_sarc = sarc_dataset.map(preprocess_function, batched=True)

In [ ]:
seq_clf_tokenized_sarc['train'][0]

In [ ]:
# DataCollatorWithPadding creates batch of data. It also dynamically pads text to the
#  length of the longest element in the batch, making them all the same length.
#  It's possible to pad your text in the tokenizer function with padding=True, dynamic padding is more efficient.

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Data Collator will pad data so that all examples are the same input length.
Attention mask is how we ignore attention scores for padding tokens

In [ ]:
sequence_clf_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-cased',
    num_labels=len(unique_sequence_labels),
)

# set an index -> label dictionary
sequence_clf_model.config.id2label = {i: l for i, l in enumerate(unique_sequence_labels)}

In [ ]:
sequence_clf_model.config.id2label[0]

In [ ]:
metric = load_metric("accuracy")

def compute_metrics(eval_pred):  # custom method to take in logits and calculate accuracy of the eval set
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
import os
import shutil

# Clear the output directory
output_dir = "./sarc/results"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
epochs = 1

training_args = TrainingArguments(
    output_dir="./sarc/results",
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,

    # some deep learning parameters that the Trainer is able to take in
    warmup_steps=len(seq_clf_tokenized_sarc['train']) // 5,  # number of warmup steps for learning rate scheduler,
    weight_decay = 0.05,

    logging_steps=1,
    log_level='info',
    evaluation_strategy='epoch',
    eval_steps=50,
    save_strategy='epoch',
    overwrite_output_dir=True
)

# Define the trainer:

trainer = Trainer(
    model=sequence_clf_model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=seq_clf_tokenized_sarc['train'],
    eval_dataset=seq_clf_tokenized_sarc['test'],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model()

In [ ]:
api_key = "<YOUR_HUGGING_FACE_TOKEN>"

MODEL_IDENTIFIER = 'brianellis1997/distilbert-sarcascm-classifier'

In [ ]:
trainer.model.push_to_hub(repo_id=MODEL_IDENTIFIER, use_auth_token=api_key)

In [ ]:
# Load the tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-cased')

# Push tokenizer to the Hugging Face model hub
tokenizer.push_to_hub(repo_id=MODEL_IDENTIFIER, use_auth_token=api_key)

# Continue Training (DistilBert)

In [ ]:
!pip install datasets

In [ ]:
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, DistilBertTokenizerFast, \
     DataCollatorWithPadding, pipeline, AutoTokenizer, AutoModelForSequenceClassification

from datasets import load_metric, Dataset
import pandas as pd
import numpy as np

import requests

In [ ]:
api_key = "<YOUR_HUGGING_FACE_TOKEN>"

MODEL_IDENTIFIER = 'brianellis1997/distilbert-sarcascm-classifier'

In [ ]:
# Load up our most recent version
auto_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDENTIFIER)
auto_model = AutoModelForSequenceClassification.from_pretrained(MODEL_IDENTIFIER)

In [ ]:
sarc_clf = pipeline(
    'text-classification',
    model=auto_model,
    tokenizer=auto_tokenizer,
    use_fast=True,               # Use a fast tokenizer if possible
    return_all_scores=True       # Return probabilities for all classes
)

In [ ]:
sarc_clf('parent_comment: Lovely weather today. child_comment: Yeah, I love when it rains.')

In [ ]:
# conjoin comment and parent comment
train_bal['joined_comment'] = 'parent_comment: ' + train_bal['parent_comment'] + ' child_comment: ' + train_bal['comment']
# here we are concatenating the comment to the parent comment and providing context for our LLM model by prompting the parent comment and child comment locations.

train_bal.head()

In [ ]:
# This code segment parses our dataset into a more manageable format

utterances = train_bal['joined_comment']
sequence_labels = train_bal['label']

In [ ]:
len(utterances), len(sequence_labels)

In [ ]:
unique_sequence_labels = list(set(sequence_labels))
unique_sequence_labels

In [ ]:
sequence_labels = [unique_sequence_labels.index(l) for l in sequence_labels]

print(f'There are {len(unique_sequence_labels)} unique sequence labels')

In [ ]:
sarc_dataset = Dataset.from_dict(
    dict(
        utterance=utterances,
        label=sequence_labels
    )
)
sarc_dataset = sarc_dataset.train_test_split(test_size=0.2)

In [ ]:
sarc_dataset['train'][0]

In [ ]:
# simple function to batch tokenize utterances with truncation
def preprocess_function(examples):
    return auto_tokenizer(examples["utterance"], truncation=True)  # Truncation will solve the problem of longer than 512 length tokens

In [ ]:
seq_clf_tokenized_sarc = sarc_dataset.map(preprocess_function, batched=True)

In [ ]:
# DataCollatorWithPadding creates batch of data. It also dynamically pads text to the
#  length of the longest element in the batch, making them all the same length.
#  It's possible to pad your text in the tokenizer function with padding=True, dynamic padding is more efficient.

data_collator = DataCollatorWithPadding(tokenizer=auto_tokenizer)

Data Collator will pad data so that all examples are the same input length.
Attention mask is how we ignore attention scores for padding tokens

In [ ]:
metric = load_metric("accuracy")

def compute_metrics(eval_pred):  # custom method to take in logits and calculate accuracy of the eval set
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
import os
import shutil

# Clear the output directory
output_dir = "./sarc/results"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
from transformers import EarlyStoppingCallback

epochs = 4

# Your existing TrainingArguments configuration
training_args = TrainingArguments(
    output_dir="./sarc/results",
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    warmup_steps=len(seq_clf_tokenized_sarc['train']) // 5,
    weight_decay=0.05,
    logging_steps=1,
    log_level='info',
    evaluation_strategy='epoch',
    eval_steps=50,
    save_strategy='epoch',
    overwrite_output_dir=True
)

# Define the Trainer with the early stopping callback
trainer = Trainer(
    model=auto_model,
    tokenizer=auto_tokenizer,
    args=training_args,
    train_dataset=seq_clf_tokenized_sarc['train'],
    eval_dataset=seq_clf_tokenized_sarc['test'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  # Early stopping after 1 epoch without improvement
)

In [ ]:
trainer.evaluate()

In [ ]:
trainer.train()

In [ ]:
trainer.save_model()

In [ ]:
trainer.model.push_to_hub(repo_id=MODEL_IDENTIFIER, use_auth_token=api_key)

In [ ]:
# Push tokenizer to the Hugging Face model hub
auto_tokenizer.push_to_hub(repo_id=MODEL_IDENTIFIER, use_auth_token=api_key)

# Obtaining Final Validation Accuracy

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
!pip install datasets

In [ ]:
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, DistilBertTokenizerFast, \
     DataCollatorWithPadding, pipeline, AutoTokenizer, AutoModelForSequenceClassification

from datasets import load_metric, Dataset
import pandas as pd
import numpy as np

import requests

In [ ]:
api_key = "<YOUR_HUGGING_FACE_TOKEN>"

MODEL_IDENTIFIER = 'brianellis1997/distilbert-sarcascm-classifier'

In [ ]:
# Load up our most recent version
auto_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDENTIFIER)
auto_model = AutoModelForSequenceClassification.from_pretrained(MODEL_IDENTIFIER)

In [ ]:
sarc_clf = pipeline(
    'text-classification',
    model=auto_model,
    tokenizer=auto_tokenizer,
    use_fast=True,               # Use a fast tokenizer if possible
    return_all_scores=True       # Return probabilities for all classes
)

In [ ]:
sarc_clf('parent_comment: Lovely weather today. child_comment: Yeah, I love when it rains.')

In [ ]:
# conjoin comment and parent comment
train_bal['joined_comment'] = 'parent_comment: ' + train_bal['parent_comment'] + ' child_comment: ' + train_bal['comment']
# here we are concatenating the comment to the parent comment and providing context for our LLM model by prompting the parent comment and child comment locations.

train_bal.head()

In [ ]:
# This code segment parses our dataset into a more manageable format

utterances = train_bal['joined_comment']
sequence_labels = train_bal['label']

In [ ]:
len(utterances), len(sequence_labels)

In [ ]:
unique_sequence_labels = list(set(sequence_labels))
unique_sequence_labels

In [ ]:
sequence_labels = [unique_sequence_labels.index(l) for l in sequence_labels]

print(f'There are {len(unique_sequence_labels)} unique sequence labels')

In [ ]:
sarc_dataset = Dataset.from_dict(
    dict(
        utterance=utterances,
        label=sequence_labels
    )
)
sarc_dataset = sarc_dataset.train_test_split(test_size=0.2)

In [ ]:
sarc_dataset['train'][0]

In [ ]:
# simple function to batch tokenize utterances with truncation
def preprocess_function(examples):
    return auto_tokenizer(examples["utterance"], truncation=True)  # Truncation will solve the problem of longer than 512 length tokens

In [ ]:
seq_clf_tokenized_sarc = sarc_dataset.map(preprocess_function, batched=True)

In [ ]:
# DataCollatorWithPadding creates batch of data. It also dynamically pads text to the
#  length of the longest element in the batch, making them all the same length.
#  It's possible to pad your text in the tokenizer function with padding=True, dynamic padding is more efficient.

data_collator = DataCollatorWithPadding(tokenizer=auto_tokenizer)

Data Collator will pad data so that all examples are the same input length.
Attention mask is how we ignore attention scores for padding tokens

In [ ]:
metric = load_metric("accuracy")

def compute_metrics(eval_pred):  # custom method to take in logits and calculate accuracy of the eval set
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
import os
import shutil

# Clear the output directory
output_dir = "./sarc/results"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
from transformers import EarlyStoppingCallback

epochs = 1

# Your existing TrainingArguments configuration
training_args = TrainingArguments(
    output_dir="./sarc/results",
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    warmup_steps=len(seq_clf_tokenized_sarc['train']) // 5,
    weight_decay=0.05,
    logging_steps=1,
    log_level='info',
    evaluation_strategy='epoch',
    eval_steps=50,
    save_strategy='epoch',
    overwrite_output_dir=True
)

# Define the Trainer with the early stopping callback
trainer = Trainer(
    model=auto_model,
    tokenizer=auto_tokenizer,
    args=training_args,
    train_dataset=seq_clf_tokenized_sarc['train'],
    eval_dataset=seq_clf_tokenized_sarc['test'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  # Early stopping after 1 epoch without improvement
)

In [ ]:
trainer.evaluate()

Our final validation accuracy is 81.5%

## Test Accuracy

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
# !pip install datasets

In [ ]:
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, DistilBertTokenizerFast, \
     DataCollatorWithPadding, pipeline, AutoTokenizer, AutoModelForSequenceClassification

from datasets import load_metric, Dataset
import pandas as pd
import numpy as np

import requests

In [ ]:
test_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')
test_bal.head()

In [ ]:
api_key = "<YOUR_HUGGING_FACE_TOKEN>"

MODEL_IDENTIFIER = 'brianellis1997/distilbert-sarcascm-classifier'

In [ ]:
# Load up our most recent version
auto_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDENTIFIER)
auto_model = AutoModelForSequenceClassification.from_pretrained(MODEL_IDENTIFIER)

In [ ]:
sarc_clf = pipeline(
    'text-classification',
    model=auto_model,
    tokenizer=auto_tokenizer,
    use_fast=True,               # Use a fast tokenizer if possible
    return_all_scores=True       # Return probabilities for all classes
)

In [ ]:
sarc_clf('parent_comment: Lovely weather today. child_comment: Yeah, I love when it rains.')

In [ ]:
# conjoin comment and parent comment
test_bal['joined_comment'] = 'parent_comment: ' + test_bal['parent_comment'] + ' child_comment: ' + test_bal['comment']
# here we are concatenating the comment to the parent comment and providing context for our LLM model by prompting the parent comment and child comment locations.

test_bal['joined_comment'].head()

In [ ]:
# This code segment parses our dataset into a more manageable format

utterances = test_bal['joined_comment']
sequence_labels = test_bal['label']

In [ ]:
len(utterances), len(sequence_labels)

In [ ]:
unique_sequence_labels = list(set(sequence_labels))
unique_sequence_labels

In [ ]:
sequence_labels = [unique_sequence_labels.index(l) for l in sequence_labels]

print(f'There are {len(unique_sequence_labels)} unique sequence labels')

In [ ]:
test_dataset = Dataset.from_dict(
    dict(
        utterance=utterances,
        label=sequence_labels
    )
)

In [ ]:
# simple function to batch tokenize utterances with truncation
def preprocess_function(examples):
    return auto_tokenizer(examples["utterance"], truncation=True)  # Truncation will solve the problem of longer than 512 length tokens

In [ ]:
# Apply the preprocessing function you defined earlier
test_dataset = test_dataset.map(preprocess_function, batched=True)

In [ ]:
# Make sure your dataset has the correct format for evaluation
test_dataset = test_dataset.map(lambda examples: {'labels': examples['label']}, batched=True)

In [ ]:
# DataCollatorWithPadding creates batch of data. It also dynamically pads text to the
#  length of the longest element in the batch, making them all the same length.
#  It's possible to pad your text in the tokenizer function with padding=True, dynamic padding is more efficient.

data_collator = DataCollatorWithPadding(tokenizer=auto_tokenizer)

Data Collator will pad data so that all examples are the same input length.
Attention mask is how we ignore attention scores for padding tokens

In [ ]:
metric = load_metric("accuracy")

def compute_metrics(eval_pred):  # custom method to take in logits and calculate accuracy of the eval set
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
import os
import shutil

# Clear the output directory
output_dir = "./sarc/results"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# !pip install transformers==4.28.0

In [ ]:
from transformers import EarlyStoppingCallback

epochs = 1

# Your existing TrainingArguments configuration
training_args = TrainingArguments(
    output_dir="./sarc/results",
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    weight_decay=0.05,
    logging_steps=1,
    log_level='info',
    evaluation_strategy='epoch',
    eval_steps=50,
    save_strategy='epoch',
    overwrite_output_dir=True
)

# Define the Trainer with the early stopping callback
trainer = Trainer(
    model=auto_model,
    tokenizer=auto_tokenizer,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [ ]:
trainer.evaluate()

Our test accuracy for our DistilBert model is 77.2% accuracy.